# 腕立て伏せ：動画から学習済みファイルを作る

## このノートブックの目的

このノートブックは、腕立て伏せの動画から、リアルタイム姿勢分析に必要な学習済みファイルを作成するためのものです。カメラ映像の表示、リアルタイム判定、改善コメントの表示、回数カウントは別のノートブックで行います。

## 全体の処理

1. KaggleからGood／Badの腕立て伏せ動画を取得します。
2. 各動画を10 FPSで読み込み、MediaPipeで肩・肘・手首・腰・膝・足首を検出します。
3. カメラから見えやすい左右どちらか一方の骨格を選びます。
4. 身体の大きさや左右の向きをそろえ、座標・関節角度・身体の傾きなど24個の特徴量を計算します。
5. 連続する15フレームを1系列として、腕立て伏せの動きを表すデータを作ります。
6. 同じ動画が複数のデータ区分に入らないよう、動画単位で学習用・検証用・テスト用に分けます。
7. 学習用データを基準に特徴量を標準化し、LSTMでGood／Badを学習します。
8. 学習用のGood動画から、改善コメントに使用する角度や動作量の基準値を計算します。
9. 学習済みモデル、標準化処理、モデル情報、改善コメント用基準値をGoogle Driveへ保存します。

## 姿勢分析側とのつながり

このノートブックは、次の4ファイルを`/content/drive/MyDrive/腕立てリアルタイム判定`へ保存します。

- `pushup_lstm_model.keras`：Good／Badを判定するLSTMモデル
- `pushup_lstm_scaler.joblib`：24個の特徴量を学習時と同じ方法で標準化する処理
- `pushup_lstm_model_info.joblib`：特徴量の名前と順序、系列長、FPS、判定しきい値などの設定
- `pushup_form_thresholds.joblib`：改善コメントに使う角度・動作量の基準値

別ファイルの`count_pushup_realtime_shared_colab(1).ipynb`は、この4ファイルを読み込んで姿勢分析を行います。そのため、ファイル名、保存先、特徴量の名前と順序は変更しないでください。

## 実行方法

Google Colabで、下のセルを上から順番に実行します。以前作成したファイルが保存先にある場合は、最後の保存処理で新しい学習結果に更新されます。


## 1. 必要なライブラリのインストール


In [ ]:
!pip -q install kagglehub mediapipe opencv-python-headless pandas numpy scikit-learn joblib tqdm tensorflow
!wget -q -O /content/pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task

print("インストールとMediaPipeモデルの準備が完了しました")


## 2. インポート・Google Drive接続・骨格検出の準備


In [ ]:
import cv2
import joblib
import kagglehub
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

drive.mount("/content/drive")

# リアルタイム姿勢分析ノートブックも、このフォルダの4ファイルを読み込みます。
OUTPUT_DIR = Path("/content/drive/MyDrive/腕立てリアルタイム判定")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_options = python.BaseOptions(
    model_asset_path="/content/pose_landmarker.task"
)
pose_options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
)
detector = vision.PoseLandmarker.create_from_options(pose_options)

print("インポート、Drive接続、骨格検出の準備が完了しました")


In [ ]:
def angle_3points(a, b, c):
    a = np.asarray(a[:2], dtype=np.float32)
    b = np.asarray(b[:2], dtype=np.float32)
    c = np.asarray(c[:2], dtype=np.float32)
    ba = a - b
    bc = c - b
    denominator = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denominator < 1e-8:
        return np.nan
    cosine = np.clip(np.dot(ba, bc) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def angle_with_horizontal(a, b):
    vector = np.asarray(a[:2], dtype=np.float32) - np.asarray(b[:2], dtype=np.float32)
    return float(np.degrees(np.arctan2(abs(vector[1]), abs(vector[0]) + 1e-8)))


def point_to_line_distance(point, line_start, line_end):
    point = np.asarray(point[:2], dtype=np.float32)
    line_start = np.asarray(line_start[:2], dtype=np.float32)
    line_end = np.asarray(line_end[:2], dtype=np.float32)
    line_vector = line_end - line_start
    denominator = np.linalg.norm(line_vector)
    if denominator < 1e-8:
        return np.nan
    relative = point - line_start
    cross_value = line_vector[0] * relative[1] - line_vector[1] * relative[0]
    return float(abs(cross_value) / denominator)


## 3. 腕立て伏せ動画の取得


In [ ]:
import kagglehub
from pathlib import Path

download_path = kagglehub.dataset_download(
    "mohamadashrafsalama/pushup"
)

DATASET_ROOT = Path(download_path)

print("ダウンロード先:")
print(DATASET_ROOT)

print("\n存在するか:")
print(DATASET_ROOT.exists())

In [ ]:
VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
}

video_paths = sorted([
    path
    for path in DATASET_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in VIDEO_EXTENSIONS
])

print("動画総数:", len(video_paths))

print("\n最初の20件:")

for path in video_paths[:20]:
    print(
        path.relative_to(DATASET_ROOT)
    )

In [ ]:
def infer_video_label(video_path):
    path_text = str(video_path).lower()

    if (
        "wrong sequence" in path_text
        or "incorrect" in path_text
        or "/wrong/" in path_text
    ):
        return "bad"

    if (
        "correct sequence" in path_text
        or "/correct/" in path_text
    ):
        return "good"

    return None

In [ ]:
video_structure_df = pd.DataFrame([
    {
        "path": str(path),
        "file_name": path.name,
        "label": infer_video_label(path),
    }
    for path in video_paths
])

video_structure_df = video_structure_df.dropna(subset=["label"]).reset_index(drop=True)
print("使用する動画数:", len(video_structure_df))


In [ ]:
metadata_rows = []

for row in tqdm(video_structure_df.itertuples(), total=len(video_structure_df)):
    path = Path(row.path)
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        capture.release()
        continue

    metadata_rows.append({
        "path": str(path),
        "file_name": path.name,
        "label": row.label,
    })
    capture.release()

video_metadata_df = pd.DataFrame(metadata_rows)
print("読み込み可能な動画数:", len(video_metadata_df))


## 4. 動画から骨格特徴量を抽出


In [ ]:
def extract_pushup_features_from_frame(
    image_bgr,
    detector
):
    if image_bgr is None:
        return None

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = detector.detect(mp_image)

    if len(result.pose_landmarks) == 0:
        return None

    landmarks = result.pose_landmarks[0]

    def get_point(index):
        landmark = landmarks[index]

        visibility = (
            landmark.visibility
            if landmark.visibility is not None
            else 0.0
        )

        return np.array(
            [
                landmark.x,
                landmark.y,
                landmark.z,
                visibility
            ],
            dtype=np.float32
        )

    left_indices = [
        11, 13, 15, 23, 25, 27
    ]

    right_indices = [
        12, 14, 16, 24, 26, 28
    ]

    left_visibility = np.mean([
        get_point(index)[3]
        for index in left_indices
    ])

    right_visibility = np.mean([
        get_point(index)[3]
        for index in right_indices
    ])

    if left_visibility >= right_visibility:
        selected_indices = left_indices
    else:
        selected_indices = right_indices

    (
        shoulder,
        elbow,
        wrist,
        hip,
        knee,
        ankle
    ) = [
        get_point(index)
        for index in selected_indices
    ]

    scale = np.linalg.norm(
        shoulder[:2] - hip[:2]
    )

    if scale < 1e-6:
        return None

    # 足首が肩より右側になるように統一
    direction = (
        1.0
        if ankle[0] >= shoulder[0]
        else -1.0
    )

    keypoints = [
        shoulder,
        elbow,
        wrist,
        hip,
        knee,
        ankle
    ]

    coordinate_features = []

    # 腰を原点にして座標を正規化
    for point in keypoints:
        normalized_x = (
            (point[0] - hip[0])
            / scale
            * direction
        )

        normalized_y = (
            point[1] - hip[1]
        ) / scale

        coordinate_features.extend([
            float(normalized_x),
            float(normalized_y)
        ])

    elbow_angle = angle_3points(
        shoulder,
        elbow,
        wrist
    )

    shoulder_angle = angle_3points(
        elbow,
        shoulder,
        hip
    )

    hip_angle = angle_3points(
        shoulder,
        hip,
        knee
    )

    knee_angle = angle_3points(
        hip,
        knee,
        ankle
    )

    torso_lean = angle_with_horizontal(
        shoulder,
        hip
    )

    lower_body_lean = angle_with_horizontal(
        hip,
        ankle
    )

    whole_body_lean = angle_with_horizontal(
        shoulder,
        ankle
    )

    hip_line_distance = (
        point_to_line_distance(
            hip,
            shoulder,
            ankle
        ) / scale
    )

    knee_line_distance = (
        point_to_line_distance(
            knee,
            shoulder,
            ankle
        ) / scale
    )

    wrist_shoulder_x = (
        (wrist[0] - shoulder[0])
        / scale
        * direction
    )

    wrist_shoulder_y = (
        wrist[1] - shoulder[1]
    ) / scale

    elbow_shoulder_y = (
        elbow[1] - shoulder[1]
    ) / scale

    features = coordinate_features + [
        elbow_angle,
        shoulder_angle,
        hip_angle,
        knee_angle,
        torso_lean,
        lower_body_lean,
        whole_body_lean,
        hip_line_distance,
        knee_line_distance,
        wrist_shoulder_x,
        wrist_shoulder_y,
        elbow_shoulder_y
    ]

    features = np.asarray(
        features,
        dtype=np.float32
    )

    if not np.all(np.isfinite(features)):
        return None

    return features.tolist()

In [ ]:
VIDEO_FEATURE_NAMES = [
    "shoulder_x",
    "shoulder_y",
    "elbow_x",
    "elbow_y",
    "wrist_x",
    "wrist_y",
    "hip_x",
    "hip_y",
    "knee_x",
    "knee_y",
    "ankle_x",
    "ankle_y",

    "elbow_angle",
    "shoulder_angle",
    "hip_angle",
    "knee_angle",

    "torso_lean",
    "lower_body_lean",
    "whole_body_lean",

    "hip_line_distance",
    "knee_line_distance",

    "wrist_shoulder_x",
    "wrist_shoulder_y",
    "elbow_shoulder_y"
]

print(
    "1フレームの特徴量数:",
    len(VIDEO_FEATURE_NAMES)
)

In [ ]:
from tqdm.auto import tqdm

TARGET_FPS = 10.0

video_feature_rows = []
video_processing_rows = []

for metadata_row in tqdm(
    video_metadata_df.itertuples(),
    total=len(video_metadata_df)
):
    video_path = Path(metadata_row.path)
    label = metadata_row.label

    capture = cv2.VideoCapture(
        str(video_path)
    )

    source_fps = capture.get(
        cv2.CAP_PROP_FPS
    )

    source_frame_count = int(
        capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if source_fps <= 0:
        capture.release()

        video_processing_rows.append({
            "file_name": video_path.name,
            "label": label,
            "source_frames": source_frame_count,
            "sampled_frames": 0,
            "successful_frames": 0,
            "status": "invalid_fps"
        })
        continue

    next_sample_time = 0.0
    frame_index = 0
    sampled_count = 0
    successful_count = 0

    while True:
        success, frame_bgr = capture.read()

        if not success:
            break

        current_time = (
            frame_index / source_fps
        )

        if (
            current_time + 1e-9
            >= next_sample_time
        ):
            features = (
                extract_pushup_features_from_frame(
                    frame_bgr,
                    detector
                )
            )

            sampled_count += 1

            if features is not None:
                row = {
                    "video_id": video_path.stem,
                    "file_name": video_path.name,
                    "path": str(video_path),
                    "label": label,
                    "time_seconds": current_time,
                    "source_frame": frame_index,
                    "sequence_index": sampled_count - 1
                }

                for feature_name, value in zip(
                    VIDEO_FEATURE_NAMES,
                    features
                ):
                    row[feature_name] = value

                video_feature_rows.append(row)
                successful_count += 1

            next_sample_time += (
                1.0 / TARGET_FPS
            )

        frame_index += 1

    capture.release()

    video_processing_rows.append({
        "file_name": video_path.name,
        "label": label,
        "source_frames": source_frame_count,
        "sampled_frames": sampled_count,
        "successful_frames": successful_count,
        "status": "completed"
    })

video_frame_df = pd.DataFrame(
    video_feature_rows
)

video_processing_df = pd.DataFrame(
    video_processing_rows
)

print("処理完了")
print("抽出成功フレーム総数:", len(video_frame_df))

print("\nラベル別の成功フレーム数:")
print(
    video_frame_df["label"].value_counts()
)

print("\n成功した動画数:")
print(
    video_frame_df.groupby("label")[
        "video_id"
    ].nunique()
)

In [ ]:
VIDEO_FEATURE_PATH = OUTPUT_DIR / "pushup_video_frame_features_10fps.csv"
VIDEO_PROCESSING_PATH = OUTPUT_DIR / "pushup_video_processing_summary.csv"

video_frame_df.to_csv(VIDEO_FEATURE_PATH, index=False)
video_processing_df.to_csv(VIDEO_PROCESSING_PATH, index=False)

print("骨格特徴量と処理結果を保存しました")
print(VIDEO_FEATURE_PATH)
print(VIDEO_PROCESSING_PATH)


## 5. 15フレームの系列データを作成し、動画単位で分割


In [ ]:
video_frame_df["video_key"] = (
    video_frame_df["label"]
    + "__"
    + video_frame_df["video_id"]
)


In [ ]:
SEQUENCE_LENGTH = 15
SEQUENCE_STEP = 5

sequence_arrays = []
sequence_labels = []
sequence_video_keys = []
sequence_start_times = []
sequence_end_times = []

skipped_noncontinuous = 0

for video_key, one_video_df in (
    video_frame_df.groupby("video_key")
):
    one_video_df = (
        one_video_df
        .sort_values("sequence_index")
        .reset_index(drop=True)
    )

    label = one_video_df["label"].iloc[0]

    number_of_rows = len(one_video_df)

    if number_of_rows < SEQUENCE_LENGTH:
        continue

    for start_index in range(
        0,
        number_of_rows - SEQUENCE_LENGTH + 1,
        SEQUENCE_STEP
    ):
        end_index = (
            start_index + SEQUENCE_LENGTH
        )

        window_df = one_video_df.iloc[
            start_index:end_index
        ]

        sampled_indices = window_df[
            "sequence_index"
        ].to_numpy()

        # 検出失敗などで番号が飛んでいないか確認
        index_differences = np.diff(
            sampled_indices
        )

        if not np.all(
            index_differences == 1
        ):
            skipped_noncontinuous += 1
            continue

        feature_array = window_df[
            VIDEO_FEATURE_NAMES
        ].to_numpy(
            dtype=np.float32
        )

        sequence_arrays.append(
            feature_array
        )

        sequence_labels.append(label)
        sequence_video_keys.append(video_key)

        sequence_start_times.append(
            window_df["time_seconds"].iloc[0]
        )

        sequence_end_times.append(
            window_df["time_seconds"].iloc[-1]
        )

X_sequences = np.stack(
    sequence_arrays
)

y_sequences = np.asarray(
    sequence_labels
)

sequence_video_keys_array = np.asarray(
    sequence_video_keys
)

sequence_metadata_df = pd.DataFrame({
    "video_key": sequence_video_keys,
    "label": sequence_labels,
    "start_time": sequence_start_times,
    "end_time": sequence_end_times
})

print("系列データの形:")
print(X_sequences.shape)

print(
    "\n意味: サンプル数 × 15フレーム × 24特徴量"
)

print("\nラベル別の系列数:")
print(
    sequence_metadata_df[
        "label"
    ].value_counts()
)

print("\nラベル別の動画数:")
print(
    sequence_metadata_df
    .groupby("label")["video_key"]
    .nunique()
)

print(
    "\nフレーム欠損により除外した系列:",
    skipped_noncontinuous
)

In [ ]:
from sklearn.model_selection import train_test_split

video_label_df = (
    sequence_metadata_df[
        ["video_key", "label"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 70%を学習、30%を一時データ
train_video_df, temporary_video_df = (
    train_test_split(
        video_label_df,
        test_size=0.30,
        random_state=42,
        stratify=video_label_df["label"]
    )
)

# 一時データを検証用とテスト用に半分ずつ分割
validation_video_df, test_video_df = (
    train_test_split(
        temporary_video_df,
        test_size=0.50,
        random_state=42,
        stratify=temporary_video_df["label"]
    )
)

train_video_keys = set(
    train_video_df["video_key"]
)

validation_video_keys = set(
    validation_video_df["video_key"]
)

test_video_keys = set(
    test_video_df["video_key"]
)


In [ ]:
train_mask = (
    sequence_metadata_df["video_key"]
    .isin(train_video_keys)
    .to_numpy()
)

validation_mask = (
    sequence_metadata_df["video_key"]
    .isin(validation_video_keys)
    .to_numpy()
)

test_mask = (
    sequence_metadata_df["video_key"]
    .isin(test_video_keys)
    .to_numpy()
)

X_train = X_sequences[train_mask]
X_validation = X_sequences[validation_mask]
X_test = X_sequences[test_mask]

# badを1、goodを0に変換
y_binary = np.where(
    y_sequences == "bad",
    1,
    0
).astype(np.int32)

y_train = y_binary[train_mask]
y_validation = y_binary[validation_mask]
y_test = y_binary[test_mask]

test_metadata_df = (
    sequence_metadata_df.loc[test_mask]
    .reset_index(drop=True)
)

print("学習系列:", X_train.shape)
print("検証系列:", X_validation.shape)
print("テスト系列:", X_test.shape)

print("\n学習ラベル:")
print(
    pd.Series(y_train)
    .map({0: "good", 1: "bad"})
    .value_counts()
)

print("\n検証ラベル:")
print(
    pd.Series(y_validation)
    .map({0: "good", 1: "bad"})
    .value_counts()
)

print("\nテストラベル:")
print(
    pd.Series(y_test)
    .map({0: "good", 1: "bad"})
    .value_counts()
)

In [ ]:
from sklearn.preprocessing import StandardScaler

number_of_features = X_train.shape[2]

scaler = StandardScaler()

# 学習データだけで平均と標準偏差を計算
scaler.fit(
    X_train.reshape(
        -1,
        number_of_features
    )
)

X_train_scaled = scaler.transform(
    X_train.reshape(
        -1,
        number_of_features
    )
).reshape(X_train.shape)

X_validation_scaled = scaler.transform(
    X_validation.reshape(
        -1,
        number_of_features
    )
).reshape(X_validation.shape)

X_test_scaled = scaler.transform(
    X_test.reshape(
        -1,
        number_of_features
    )
).reshape(X_test.shape)

print("標準化完了")
print(X_train_scaled.shape)

## 6. LSTMを学習


In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(
            SEQUENCE_LENGTH,
            len(VIDEO_FEATURE_NAMES)
        )
    ),

    tf.keras.layers.LSTM(
        48,
        return_sequences=True
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.LSTM(24),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(
        16,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
from sklearn.utils.class_weight import (
    compute_class_weight
)

classes = np.array([0, 1])

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    0: class_weight_values[0],
    1: class_weight_values[1]
}

print("クラス重み:", class_weights)

In [ ]:
early_stopping = (
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(
        X_validation_scaled,
        y_validation
    ),
    epochs=100,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

## 7. 改善コメントに使う角度等の基準値を計算


In [ ]:
def summarize_pushup_sequence(
    sequence,
    feature_names
):
    feature_index = {
        name: index
        for index, name in enumerate(
            feature_names
        )
    }

    def values(name):
        return sequence[
            :,
            feature_index[name]
        ]

    elbow_angle = values("elbow_angle")
    hip_angle = values("hip_angle")
    knee_angle = values("knee_angle")

    hip_line = values(
        "hip_line_distance"
    )

    knee_line = values(
        "knee_line_distance"
    )

    shoulder_y = values("shoulder_y")
    wrist_shoulder_y = values(
        "wrist_shoulder_y"
    )

    return {
        "minimum_elbow_angle":
            float(np.min(elbow_angle)),

        "maximum_elbow_angle":
            float(np.max(elbow_angle)),

        "elbow_angle_range":
            float(
                np.max(elbow_angle)
                - np.min(elbow_angle)
            ),

        "minimum_hip_angle":
            float(np.min(hip_angle)),

        "mean_hip_angle":
            float(np.mean(hip_angle)),

        "minimum_knee_angle":
            float(np.min(knee_angle)),

        "maximum_hip_line_distance":
            float(np.max(hip_line)),

        "mean_hip_line_distance":
            float(np.mean(hip_line)),

        "maximum_knee_line_distance":
            float(np.max(knee_line)),

        "shoulder_movement":
            float(
                np.max(shoulder_y)
                - np.min(shoulder_y)
            ),

        "wrist_shoulder_y_range":
            float(
                np.max(wrist_shoulder_y)
                - np.min(wrist_shoulder_y)
            )
    }

In [ ]:
summary_rows = []

for index, sequence in enumerate(
    X_sequences
):
    summary = summarize_pushup_sequence(
        sequence,
        VIDEO_FEATURE_NAMES
    )

    summary["video_key"] = (
        sequence_metadata_df.iloc[index][
            "video_key"
        ]
    )

    summary["label"] = (
        sequence_metadata_df.iloc[index][
            "label"
        ]
    )

    if summary["video_key"] in train_video_keys:
        summary["split"] = "train"
    elif summary["video_key"] in validation_video_keys:
        summary["split"] = "validation"
    elif summary["video_key"] in test_video_keys:
        summary["split"] = "test"
    else:
        summary["split"] = "unknown"

    summary_rows.append(summary)

sequence_summary_df = pd.DataFrame(
    summary_rows
)

print(sequence_summary_df.shape)
display(sequence_summary_df.head())

In [ ]:
SUMMARY_FEATURES = [
    "minimum_elbow_angle",
    "maximum_elbow_angle",
    "elbow_angle_range",
    "minimum_hip_angle",
    "mean_hip_angle",
    "minimum_knee_angle",
    "maximum_hip_line_distance",
    "mean_hip_line_distance",
    "maximum_knee_line_distance",
    "shoulder_movement",
    "wrist_shoulder_y_range",
]

training_summary_df = sequence_summary_df[
    sequence_summary_df["split"] == "train"
].copy()


In [ ]:
training_good_df = (
    training_summary_df[
        training_summary_df["label"]
        == "good"
    ]
)

form_thresholds = {
    # Goodより肘の最小角度が大きい
    # ＝十分に身体を下げていない可能性
    "shallow_elbow_angle": float(
        training_good_df[
            "minimum_elbow_angle"
        ].quantile(0.90)
    ),

    # Goodより曲げ伸ばしの幅が小さい
    "small_elbow_range": float(
        training_good_df[
            "elbow_angle_range"
        ].quantile(0.10)
    ),

    # Goodより腰角度が小さい
    "small_hip_angle": float(
        training_good_df[
            "minimum_hip_angle"
        ].quantile(0.10)
    ),

    # Goodより膝角度が小さい
    "small_knee_angle": float(
        training_good_df[
            "minimum_knee_angle"
        ].quantile(0.10)
    ),

    # Goodより腰が身体の直線から外れる
    "large_hip_line_distance": float(
        training_good_df[
            "maximum_hip_line_distance"
        ].quantile(0.90)
    ),

    # Goodより肩の上下動が小さい
    "small_shoulder_movement": float(
        training_good_df[
            "shoulder_movement"
        ].quantile(0.10)
    )
}

print("学習用Goodから決めた基準値:")

for name, value in form_thresholds.items():
    print(f"{name}: {value:.4f}")

## 8. リアルタイム姿勢分析用の4ファイルを保存


In [ ]:
LSTM_MODEL_PATH = OUTPUT_DIR / "pushup_lstm_model.keras"
SCALER_PATH = OUTPUT_DIR / "pushup_lstm_scaler.joblib"
MODEL_INFO_PATH = OUTPUT_DIR / "pushup_lstm_model_info.joblib"
THRESHOLD_PATH = OUTPUT_DIR / "pushup_form_thresholds.joblib"

model.save(LSTM_MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(form_thresholds, THRESHOLD_PATH)

model_info = {
    "feature_names": VIDEO_FEATURE_NAMES,
    "sequence_length": SEQUENCE_LENGTH,
    "target_fps": TARGET_FPS,
    "input_shape": (SEQUENCE_LENGTH, len(VIDEO_FEATURE_NAMES)),
    "good_label": 0,
    "bad_label": 1,
    "classification_threshold": 0.5,
    "maximum_feedback_messages": 2,
    "model_type": "LSTM",
    "description": "Push-up Good/Bad classifier using 15 consecutive MediaPipe pose frames",
}
joblib.dump(model_info, MODEL_INFO_PATH)

saved_paths = [
    LSTM_MODEL_PATH,
    SCALER_PATH,
    MODEL_INFO_PATH,
    THRESHOLD_PATH,
]

print("保存完了")
for path in saved_paths:
    print(path.name, "→", "存在します" if path.exists() else "見つかりません")


## 出力と姿勢分析ノートブックの接続

`/content/drive/MyDrive/腕立てリアルタイム判定`に次の4ファイルが作られます。

- `pushup_lstm_model.keras`
- `pushup_lstm_scaler.joblib`
- `pushup_lstm_model_info.joblib`
- `pushup_form_thresholds.joblib`

`count_pushup_realtime_shared_colab(1).ipynb`は同じフォルダ名・ファイル名を使うため、学習後にファイルを移動したり名前を変更したりする必要はありません。骨格検出、Good／Bad判定、改善コメント、回数カウント、カメラ表示は姿勢分析側で行います。


# 改善コメント用の角度・動作量の基準値の決め方

## 基本方針

固定した角度を手作業で指定するのではなく、**学習用に分けたGood動画の分布**から基準値を計算します。テスト動画を基準値作成に使わないため、テスト結果を見て基準を都合よく調整することを避けられます。

各15フレーム系列について角度や動作量を要約し、Good系列の下位10％点または上位10％点を境界として使用します。ここでいう10％点・90％点は、値を小さい順に並べたときの位置を表します。

## 保存する基準値

| 保存名 | 系列ごとに調べる値 | Good動画からの決め方 | 基準を外れた場合に想定する問題 |
|---|---|---|---|
| `shallow_elbow_angle` | 15フレーム中の肘角度の最小値 | Goodの90％点 | 最小角度がこれより大きい場合、肘が十分に曲がらず、身体を下げ切れていない可能性 |
| `small_elbow_range` | 肘角度の最大値－最小値 | Goodの10％点 | 変化量がこれより小さい場合、肘の曲げ伸ばしが不足している可能性 |
| `small_hip_angle` | 15フレーム中の腰角度の最小値 | Goodの10％点 | 最小角度がこれより小さい場合、腰が曲がっている可能性 |
| `small_knee_angle` | 15フレーム中の膝角度の最小値 | Goodの10％点 | 最小角度がこれより小さい場合、膝が曲がっている可能性 |
| `large_hip_line_distance` | 肩と足首を結ぶ直線から腰までの距離の最大値 | Goodの90％点 | 距離がこれより大きい場合、腰が身体の直線から外れている可能性 |
| `small_shoulder_movement` | 肩の正規化Y座標の最大値－最小値 | Goodの10％点 | 移動量がこれより小さい場合、身体の上下動が不足している可能性 |

## 角度の意味

- **肘角度**：肩－肘－手首の3点で、肘を頂点として計算します。腕を伸ばすと大きくなり、曲げると小さくなります。
- **腰角度**：肩－腰－膝の3点で、腰を頂点として計算します。身体が一直線に近いほど大きくなります。
- **膝角度**：腰－膝－足首の3点で、膝を頂点として計算します。脚を伸ばすほど大きくなります。
- **腰と身体ラインの距離**：肩と足首を結ぶ直線から腰がどれだけ離れているかを、肩と腰の距離で正規化した値です。撮影距離や人物の大きさの影響を小さくしています。
- **肩の移動量**：15フレーム内で肩が上下に動いた範囲です。座標は腰を原点とし、肩と腰の距離で正規化しています。

## リアルタイム姿勢分析での使われ方

LSTMがBadと判定した系列に対して、姿勢分析側が保存済みの基準値と実際の角度・動作量を比較し、該当する改善コメントを選びます。LSTMはフォーム全体がGoodかBadかを判定し、角度等の基準値はBadになった具体的な理由を説明するために使用します。

この基準値は現在の学習用Good動画を基準にした統計的なしきい値です。医学的・運動学的に唯一正しい角度を示すものではなく、学習データを変更して再学習すると値も更新されます。
